In [27]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get the current weather in a given city."""
    return f"The weather in {city} is sunny."

@tool
def send_email(to:str , subject:str , body:str):
    """send email to a user"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def delete_records(table:str , condition:str):
    """delete the records from database"""
    return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    tools = [get_weather, send_email,delete_records],
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":True,
                "delete_records":True,
                "get_weather":False
            }
        ),
    ],
    checkpointer=InMemorySaver(),
)
print("Human in the loop agent is created successfully")

Human in the loop agent is created successfully


In [29]:
config = {"configurable":{"thread_id":"3"}}

res = hitl_agent.invoke({
    "messages":{
        "role":"user",
        "content":"delete records from users where id = 1"
    }
},
config=config
                        )
print("===========agent response===========")
print(res)

===========agent response===========
{'messages': [HumanMessage(content='delete records from users where id = 1', additional_kwargs={}, response_metadata={}, id='03c190d4-c8a2-4a3b-88ba-8ef2dbfac1bc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jrgms13qw', 'function': {'arguments': '{"condition":"id = 1","table":"users"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 429, 'total_tokens': 471, 'completion_time': 0.122223566, 'completion_tokens_details': None, 'prompt_time': 0.030106915, 'prompt_tokens_details': None, 'queue_time': 0.055119355, 'total_time': 0.152330481}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a177e6-1135-776a-9ba9-5603b310c38a-0', tool_calls=[{'name': 'delete_records', 'args': {'condition': 'id = 1', 'table': 'users'},

In [30]:
approved_status = hitl_agent.invoke(
    Command(resume={"decisions": [{"type" : "approve"}]}),
    config=config
)

print("===========agent response===========")
print(approved_status)
print("===========agent response===========")
print(approved_status["messages"][-1].content)

===========agent response===========
{'messages': [HumanMessage(content='delete records from users where id = 1', additional_kwargs={}, response_metadata={}, id='03c190d4-c8a2-4a3b-88ba-8ef2dbfac1bc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jrgms13qw', 'function': {'arguments': '{"condition":"id = 1","table":"users"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 429, 'total_tokens': 471, 'completion_time': 0.122223566, 'completion_tokens_details': None, 'prompt_time': 0.030106915, 'prompt_tokens_details': None, 'queue_time': 0.055119355, 'total_time': 0.152330481}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a177e6-1135-776a-9ba9-5603b310c38a-0', tool_calls=[{'name': 'delete_records', 'args': {'condition': 'id = 1', 'table': 'users'},

In [33]:
config = {"configurable":{"thread_id":"4"},}
hitl_agent.invoke({
    "messages":{
        "role":"user",
        "content":"send an email to pramod4342@gmail.com with subject hello and body hi"
    }
},config=config)
approved_status = hitl_agent.invoke(
    Command(resume={"decisions": [{"type" : "reject"}]}),
    config=config
)
print(approved_status["messages"][-1].content)

The email was not sent — the tool call was rejected. Let me know if you'd like me to try again or make any changes before sending.
